In [0]:
# ==========================================================
# 04_hyperparameter_tuning_community.ipynb
# Random Forest Cross Validation
# Fully Serverless-Compatible Version
# ==========================================================

# -----------------------------
# 1️⃣ Imports
# -----------------------------

from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import mlflow.spark
import os
import time

# Create required Unity Catalog volumes
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.mlflow_tmp")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.sparkml_tmp")

# Required for MLflow logging
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlflow_tmp"

# Required for Spark ML internal caching (CrossValidator)
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/sparkml_tmp"

mlflow.set_experiment("/Users/ar761179@gmail.com/taxi_fare_prediction")

# -------------------------------------------------
# 3️⃣ Load Silver Dataset
# -------------------------------------------------

df = spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/silver_2019"
)

print("Total Clean Rows:", df.count())

# -------------------------------------------------
# 4️⃣ Feature Engineering (Recreate Pipeline)
# -------------------------------------------------

feature_cols = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID_idx",
    "DOLocationID_idx",
    "payment_type_idx",
    "trip_duration",
    "pickup_hour",
    "is_weekend"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data_ml = assembler.transform(df) \
                   .select(
                       "features",
                       col("fare_amount").alias("label"),
                       "pickup_month"
                   )

# -------------------------------------------------
# 5️⃣ Temporal Train-Test Split
# -------------------------------------------------

train = data_ml.filter(col("pickup_month") <= 9)
test  = data_ml.filter(col("pickup_month") > 9)

print("Train Rows:", train.count())
print("Test Rows :", test.count())

# -------------------------------------------------
# 6️⃣ Performance Configuration
# -------------------------------------------------

spark.conf.set("spark.sql.shuffle.partitions", 100)

# Use 20% sample for CV (critical for speed)
cv_train = train.sample(0.2, seed=42)

print("CV Train Rows:", cv_train.count())

# -------------------------------------------------
# 7️⃣ Random Forest Setup
# -------------------------------------------------

rf = RandomForestRegressor(
    labelCol="label",
    featuresCol="features",
    maxBins=32
)

paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 30]) \
    .addGrid(rf.maxDepth, [6, 8]) \
    .build()

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

crossval = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2
)

# -------------------------------------------------
# 8️⃣ Run Cross Validation
# -------------------------------------------------

with mlflow.start_run(run_name="RF_Tuned"):

    start = time.time()

    cv_model = crossval.fit(cv_train)

    predictions = cv_model.transform(test)

    rmse = evaluator.evaluate(predictions)

    runtime = time.time() - start

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("cv_runtime_sec", runtime)

    mlflow.spark.log_model(
        cv_model.bestModel,
        artifact_path="best_model",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

print("\n========================")
print("Tuned RF RMSE:", rmse)
print("CV Runtime (sec):", runtime)
print("========================\n")